In [1]:
import os, uuid, hashlib, platform, getpass

# 1. 填写学号
STUDENT_ID = 20244081129

# 2. 打印 conda 环境
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# 3. 机器指纹（绑定当前电脑）
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# 4. 生成唯一 ID
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

# 核对内核确实使用独立的 conda 环境
import sys
assert CONDA_ENV != "base", "请先选择独立 conda 环境的内核。"
assert os.path.isdir(os.path.join(sys.prefix, "conda-meta")), "当前内核不是 conda 环境。"
assert os.path.normcase(os.path.abspath(sys.prefix)) == os.path.normcase(
    os.path.abspath(os.environ.get("CONDA_PREFIX", ""))
), "当前内核与激活的 conda 环境不一致。"
print(f"[姓名]     宋高峰")
print(f"[Python]   {platform.python_version()}")
print(f"[解释器]   {sys.executable}")

[Conda 环境] d2l
[学号]     20244081129
[机器指纹] 002d96cc72feb8d9
[RUN_ID]   20244081129-002d96cc72feb8d9
[姓名]     宋高峰
[Python]   3.9.23
[解释器]   D:\miniconda\envs\d2l\python.exe


## 一、简单函数

分别实现问候语、字符串前后添加 goat，以及逐行重复输出。

In [2]:
def say_hi():
    print("Hello, world!")


def goat_pad(s):
    print("goat" + s + "goat")


def print_n(s, n):
    for _ in range(n):
        print(s)


print("say_hi()：")
say_hi()
print("\ngoat_pad('Python')：")
goat_pad("Python")
print("\nprint_n('学习 Python', 3)：")
print_n("学习 Python", 3)
print("\nprint_n('这行不应出现', 0)：")
print_n("这行不应出现", 0)

say_hi()：
Hello, world!

goat_pad('Python')：
goatPythongoat

print_n('学习 Python', 3)：
学习 Python
学习 Python
学习 Python

print_n('这行不应出现', 0)：


## 二、欧几里得算法

用取余运算反复更新两个数，直到余数为 0；先取绝对值以处理负数。

In [3]:
def gcd(m, n):
    m, n = abs(m), abs(n)
    while n != 0:
        m, n = n, m % n
    return m


for m, n in [(2018, 2019), (1200, 300), (5040, 60)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

print("\n负数与零的测试：")
for m, n in [(-1200, 300), (1200, -300), (-1200, -300), (0, 300), (300, 0), (0, 0)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

assert gcd(2018, 2019) == 1
assert gcd(1200, 300) == 300
assert gcd(5040, 60) == 60
assert gcd(-1200, -300) == 300

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60

负数与零的测试：
gcd(-1200, 300) = 300
gcd(1200, -300) = 300
gcd(-1200, -300) = 300
gcd(0, 300) = 300
gcd(300, 0) = 300
gcd(0, 0) = 0


负数不会改变正公约数，所以返回非负结果有意义。一个参数为 0 时返回另一个数的绝对值；`gcd(0, 0)` 按编程约定返回 0。

## 三、近似欧拉数 e

### 3.1 极限近似

计算 `(1 + 1/n)**n`，观察 n 增大时的近似结果。

In [4]:
import math


def euler_limit(n):
    if n <= 0:
        raise ValueError("n 必须是正整数")
    return float((1 + 1 / n) ** n)


print(f"参考值 e = {math.exp(1):.12f}")
print(f"{'n':>8} {'极限近似':>18} {'绝对误差':>18}")
for n in [1, 10, 100, 1000, 10000]:
    value = euler_limit(n)
    print(f"{n:8d} {value:18.12f} {abs(value - math.exp(1)):18.12f}")

参考值 e = 2.718281828459
       n               极限近似               绝对误差
       1     2.000000000000     0.718281828459
      10     2.593742460100     0.124539368359
     100     2.704813829422     0.013467999038
    1000     2.716923932236     0.001357896223
   10000     2.718145926825     0.000135901634


### 3.2 前 n 项求和

累加下标 0 到 n−1 的 `1/k!`；n 为 0 时返回 0.0。

In [5]:
def euler_infinite_sum(n):
    total = 0.0
    term = 1.0
    for k in range(n):
        if k > 0:
            term /= k
        total += term
    return float(total)


for n in [0, 1, 2, 4, 10, 15]:
    value = euler_infinite_sum(n)
    print(f"前 {n:2d} 项：{value:.12f}，类型：{type(value).__name__}")

assert euler_infinite_sum(0) == 0.0
assert euler_infinite_sum(1) == 1.0
assert math.isclose(euler_infinite_sum(4), 1 + 1 + 1 / 2 + 1 / 6)

前  0 项：0.000000000000，类型：float
前  1 项：1.000000000000，类型：float
前  2 项：2.000000000000，类型：float
前  4 项：2.666666666667，类型：float
前 10 项：2.718281525573，类型：float
前 15 项：2.718281828458，类型：float


### 3.3 按误差要求近似

用 while 循环逐项累加，直到与 `math.exp(1)` 的误差不超过 epsilon。

In [6]:
def euler_approx(epsilon):
    if not math.isfinite(epsilon) or epsilon <= 0:
        raise ValueError("epsilon 必须是有限正数")
    target = math.exp(1)
    total = 0.0
    term = 1.0
    k = 0
    while abs(target - total) > epsilon:
        new_total = total + term
        if new_total == total:
            raise ArithmeticError("要求的误差小于当前浮点累加能够达到的精度")
        total = new_total
        k += 1
        term /= k
    return float(total)


for epsilon in [1e-2, 1e-6, 1e-12]:
    value = euler_approx(epsilon)
    error = abs(value - math.exp(1))
    print(f"epsilon = {epsilon:.0e}，近似值 = {value:.15f}，误差 = {error:.3e}")
    assert error <= epsilon

epsilon = 1e-02，近似值 = 2.708333333333333，误差 = 9.948e-03
epsilon = 1e-06，近似值 = 2.718281525573192，误差 = 3.029e-07
epsilon = 1e-12，近似值 = 2.718281828458230，误差 = 8.149e-13


### 3.4 打印近似表

依次打印前 1 项到前 n 项的求和结果，每行一个。

In [7]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print_euler_sum_table(10)

1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663
2.7180555555555554
2.7182539682539684
2.71827876984127
2.7182815255731922


## 四、整数幂判断

### 4.1 判断 2 的幂

反复整除 2，最终剩下 1 则为 2 的幂；1 对应零次幂。

In [8]:
def is_power_of_2(n):
    while n > 1:
        if n % 2 != 0:
            return False
        n //= 2
    return True


for n in [1, 2, 3, 4, 8, 12, 16, 1024, 1025]:
    print(f"is_power_of_2({n}) = {is_power_of_2(n)}")

assert is_power_of_2(1) is True
assert is_power_of_2(1024) is True
assert is_power_of_2(12) is False

is_power_of_2(1) = True
is_power_of_2(2) = True
is_power_of_2(3) = False
is_power_of_2(4) = True
is_power_of_2(8) = True
is_power_of_2(12) = False
is_power_of_2(16) = True
is_power_of_2(1024) = True
is_power_of_2(1025) = False


### 4.2 判断一般底数的幂

反复整除底数 b；b 为 1 时，仅 n 为 1 成立。

In [9]:
def is_power(b, n):
    if b == 1:
        return n == 1
    while n > 1:
        if n % b != 0:
            return False
        n //= b
    return True


cases = [(2, 16), (2, 12), (3, 27), (3, 18), (5, 125), (7, 1), (1, 1), (1, 8)]
for b, n in cases:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")

assert is_power(3, 27) is True
assert is_power(3, 18) is False
assert is_power(7, 1) is True
assert is_power(1, 8) is False
assert all(is_power(2, n) == is_power_of_2(n) for n in range(1, 101))

is_power(2, 16) = True
is_power(2, 12) = False
is_power(3, 27) = True
is_power(3, 18) = False
is_power(5, 125) = True
is_power(7, 1) = True
is_power(1, 1) = True
is_power(1, 8) = False
